# Train v16 on Colab

This notebook mounts Google Drive, installs runtime dependencies, reads W&B credentials from Colab Secrets, installs the packaged version code, and launches training against the Drive-hosted market data package.

Before running this notebook, add `WANDB_API_KEY` in Colab's Secrets panel. Do not paste API keys into notebook cells.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
wandb_api_key = userdata.get('WANDB_API_KEY')
if wandb_api_key:
    os.environ['WANDB_API_KEY'] = wandb_api_key
else:
    print('WANDB_API_KEY was not found in Colab Secrets; W&B login may prompt or run offline.')


In [ ]:
!apt-get -qq update && apt-get -qq install -y graphviz
%pip install -q polars pyarrow wandb torchinfo torchview graphviz


In [ ]:
import json
import os
import shutil
import sys
import zipfile
from pathlib import Path

VERSION = 'v16'
DRIVE_ROOT = Path('/content/drive/MyDrive')
CODE_DRIVE_DIR = DRIVE_ROOT / 'quant-research-workbench' / 'colab_code' / VERSION
PACKAGE_ZIP = CODE_DRIVE_DIR / f'inhouse_transformer_{VERSION}.zip'
MANIFEST_PATH = CODE_DRIVE_DIR / 'colab_manifest.json'
CODE_ROOT = Path('/content/quant-research-workbench')

print('package:', PACKAGE_ZIP)
print('manifest:', MANIFEST_PATH)
assert PACKAGE_ZIP.exists(), f'Missing package zip: {PACKAGE_ZIP}'
assert MANIFEST_PATH.exists(), f'Missing manifest: {MANIFEST_PATH}'
manifest = json.loads(MANIFEST_PATH.read_text())
print(json.dumps(manifest, indent=2))

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)
CODE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as package:
    package.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print('installed code at', CODE_ROOT)


In [ ]:
from pathlib import Path

PROCESSED_ROOT = Path(manifest['colab_data_root'])
assert (PROCESSED_ROOT / 'bars' / '1m').exists(), f'Missing bars/1m under {PROCESSED_ROOT}'
print('processed root:', PROCESSED_ROOT)


In [ ]:
import subprocess
import sys

# Tune these in Colab before launching a long run. L4/A100 runtimes can often use larger batches.
BATCH_SIZE = 512
EPOCHS = 1
MAX_STEPS = 0  # 0 means stream the configured epoch until exhaustion.
MAX_TICKERS = 2000
EVAL_STEPS = 500
LOGGING_STEPS = 50
VALIDATION_WINDOW_COUNT = 50000
TEST_WINDOW_COUNT = 50000
ALLOW_TARGET_ACROSS_SESSION = False

train_py = CODE_ROOT / 'research' / 'inhouse_transformer' / 'v16' / 'train.py'
cmd = [
    sys.executable,
    str(train_py),
    '--processed-root', str(PROCESSED_ROOT),
    '--train-start-date', manifest['train_start_date'],
    '--train-end-date', manifest['train_end_date'],
    '--validation-start-date', manifest['validation_start_date'],
    '--validation-end-date', manifest['validation_end_date'],
    '--test-start-date', manifest['test_start_date'],
    '--test-end-date', manifest['test_end_date'],
    '--device', 'cuda',
    '--batch-size', str(BATCH_SIZE),
    '--epochs', str(EPOCHS),
    '--max-tickers', str(MAX_TICKERS),
    '--eval-steps', str(EVAL_STEPS),
    '--logging-steps', str(LOGGING_STEPS),
    '--validation-window-count', str(VALIDATION_WINDOW_COUNT),
    '--test-window-count', str(TEST_WINDOW_COUNT),
    '--wandb-entity', manifest['wandb_entity'],
    '--wandb-project', manifest['wandb_project'],
]
if MAX_STEPS > 0:
    cmd += ['--max-steps', str(MAX_STEPS)]
if ALLOW_TARGET_ACROSS_SESSION:
    cmd.append('--allow-target-across-session')

print(' '.join(cmd))
subprocess.run(cmd, check=True)
